In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score


try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [16]:
train_main = pd.read_parquet('./data/train_main_features.parquet')
test_main = pd.read_parquet('./data/test_main_features.parquet')
target = pd.read_parquet('data/train_target.parquet')
train_extra = pd.read_parquet('./data/train_extra_features.parquet')
test_extra = pd.read_parquet('./data/test_extra_features.parquet')


print('Тренировочные данные:', train_main.shape)
print('Тестовые данные:', test_main.shape)
print('Тренировочные данные:', train_extra.shape)
print('Тестовые данные:', test_extra.shape)

Тренировочные данные: (750000, 200)
Тестовые данные: (250000, 200)
Тренировочные данные: (750000, 2242)
Тестовые данные: (250000, 2242)


In [17]:
cat_feature_names = [
    col_name for col_name in train_main.columns 
    if col_name.startswith("cat_feature")
]

In [18]:
train = pd.merge(train_main, train_extra, on="customer_id", how="left")
test  = pd.merge(test_main,  test_extra,  on="customer_id", how="left")

In [20]:
train[cat_feature_names] = train[cat_feature_names].astype(str)
test[cat_feature_names] = test[cat_feature_names].astype(str)

In [23]:
train_pool = Pool(data = train.drop("customer_id", axis=1), 
                  label = target.drop("customer_id", axis=1), 
                  cat_features = cat_feature_names)

In [ ]:
model = CatBoostClassifier(
    iterations=10000,
    depth=6,
    learning_rate=0.001,
    loss_function="MultiLogloss",
    random_seed=42,
    verbose=100,
    task_type="GPU",
    l2_leaf_reg = 10,
    devices='0',

)
model.fit(train_pool)

0:	learn: 0.6907132	total: 539ms	remaining: 1h 29m 47s
100:	learn: 0.4914477	total: 53.9s	remaining: 1h 28m 4s
200:	learn: 0.3618147	total: 1m 49s	remaining: 1h 28m 39s
300:	learn: 0.2781284	total: 2m 44s	remaining: 1h 28m 29s
400:	learn: 0.2232739	total: 3m 40s	remaining: 1h 28m 4s
500:	learn: 0.1863756	total: 4m 36s	remaining: 1h 27m 20s


In [ ]:
predict_schema = [col.replace("target_", "predict_") for col in target.columns if col.startswith("target_")]

catboost_predictions = pl.DataFrame(test_predict, schema = predict_schema)

catboost_predictions.head(n = 5)

проверим roc_auc на тренировочной выборке

In [ ]:
y_true = target.drop('customer_id').to_pandas()
y_pred = model.predict(train_pool, prediction_type = 'RawFormulaVal')
# roc_auc_score(y_true, y_pred, average="macro")

roc_auc_score(y_true, y_pred, average="macro")

сохраняем сабмит для сдачи

In [ ]:
submit = test.select("customer_id")

submit = submit.hstack(catboost_predictions)

In [ ]:
submit.write_parquet("./submits/exp_1_2.parquet")

сохраняем модельку))))))

In [ ]:
model.save_model("./models/exp1_2.cbm")